# ONNX benchmark on an x86 CPU (Colab)

Part of [onnx-inference-optimization](https://github.com/abhisheklakhani-it/onnx-inference-optimization).
Most servers and Docker hosts run Linux on x86 CPUs, so this repeats the Mac benchmark there.

1. **Runtime → Change runtime type → CPU**, then **Runtime → Run all**.
2. When asked, upload `day5_distilbert.zip` (the Day 5 `models/distilbert/` folder, zipped).
   Every step stops the notebook if it fails, so a failed run never downloads an empty zip.
3. The run takes about 25 minutes. The last cell downloads `onnx_x86_results.zip`;
   unzip it into the project folder (it only contains `results/colab_x86/`).

In [ ]:
!lscpu | grep -E 'Model name|^CPU\(s\)|Thread|Flags' | cut -c1-200

In [ ]:
!pip -q install torch==2.14.1 transformers==5.19.0 onnxruntime==1.31.0 onnx==1.23.2 onnxscript==0.7.2 tokenizers==0.23.3 datasets==5.1.0 scikit-learn
assert _exit_code == 0, "this step failed: see the error above"
!python -c 'import torch, onnxruntime, transformers; print(torch.__version__, onnxruntime.__version__, transformers.__version__)'

In [ ]:
!mkdir -p src/onnxopt scripts results models

In [ ]:
%%writefile src/onnxopt/__init__.py
"""Export the Day 5 German DistilBERT to ONNX, optimize and quantize it, benchmark, and serve it."""

In [ ]:
%%writefile src/onnxopt/paths.py
"""Where the source model comes from and where every variant is written."""

import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]

# The fine-tuned DistilBERT from Day 5 (german-text-classifier). Override with DAY5_MODEL_DIR.
SOURCE_MODEL = Path(os.environ.get("DAY5_MODEL_DIR", ROOT.parent / "german-text-classifier" / "models" / "distilbert"))

MODELS = Path(os.environ.get("MODELS_DIR", ROOT / "models"))
RESULTS = ROOT / "results"
DOCS = ROOT / "docs"

# variant name -> ONNX file in MODELS ("pytorch" is the source model itself)
ONNX_FILES = {
    "onnx_fp32": "model.onnx",
    "onnx_optimized": "model.optimized.onnx",
    "onnx_int8": "model.optimized.int8.onnx",
}
VARIANTS = ["pytorch", *ONNX_FILES]
DISPLAY = {
    "pytorch": "PyTorch fp32",
    "onnx_fp32": "ONNX fp32",
    "onnx_optimized": "ONNX optimized",
    "onnx_int8": "ONNX optimized + int8",
}

In [ ]:
%%writefile src/onnxopt/classifier.py
"""Classify German news text with an ONNX model: onnxruntime + tokenizers only, no PyTorch.

This is what the API and the Docker image use. The folder must hold the .onnx file plus
tokenizer.json and config.json copied from the source model (scripts/export.py does that).
"""

import json
from pathlib import Path

import numpy as np
import onnxruntime as ort
from tokenizers import Tokenizer


def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


def load_tokenizer(model_dir: Path, max_length: int = 512) -> Tokenizer:
    tokenizer = Tokenizer.from_file(str(model_dir / "tokenizer.json"))
    tokenizer.enable_truncation(max_length)
    tokenizer.enable_padding(pad_id=tokenizer.token_to_id("[PAD]"), pad_token="[PAD]")
    return tokenizer


def encode(tokenizer: Tokenizer, texts: list[str]) -> dict[str, np.ndarray]:
    batch = tokenizer.encode_batch(texts)
    return {
        "input_ids": np.array([e.ids for e in batch], dtype=np.int64),
        "attention_mask": np.array([e.attention_mask for e in batch], dtype=np.int64),
    }


class OnnxClassifier:
    def __init__(self, model_path: Path, threads: int | None = None, max_length: int = 512):
        model_dir = model_path.parent
        options = ort.SessionOptions()
        if threads:
            options.intra_op_num_threads = threads
        self.session = ort.InferenceSession(str(model_path), options, providers=["CPUExecutionProvider"])
        self.tokenizer = load_tokenizer(model_dir, max_length)
        config = json.loads((model_dir / "config.json").read_text())
        self.labels = [config["id2label"][str(i)] for i in range(len(config["id2label"]))]

    def logits(self, texts: list[str]) -> np.ndarray:
        return self.session.run(["logits"], encode(self.tokenizer, texts))[0]

    def predict(self, texts: list[str]) -> list[dict]:
        probs = softmax(self.logits(texts))
        return [{"label": self.labels[int(p.argmax())], "confidence": round(float(p.max()), 4)} for p in probs]

In [ ]:
%%writefile src/onnxopt/data.py
"""The official 10kGNAD test split (1,028 articles), the same one Day 5 reported on."""

DATASET = "community-datasets/gnad10"


def load_test() -> tuple[list[str], list[int], list[str]]:
    from datasets import load_dataset

    test = load_dataset(DATASET, split="test")
    return list(test["text"]), list(test["label"]), test.features["label"].names

In [ ]:
%%writefile src/onnxopt/export.py
"""Export a Hugging Face sequence classifier to ONNX with dynamic batch and sequence length."""

from pathlib import Path

OPSET = 17


def export_onnx(model, sample: dict, path: Path, opset: int = OPSET) -> None:
    """`sample` is a tokenized example batch (input_ids, attention_mask as torch tensors) used for tracing."""
    import torch

    torch.onnx.export(
        model.eval(),
        (sample["input_ids"], sample["attention_mask"]),
        str(path),
        input_names=["input_ids", "attention_mask"],
        output_names=["logits"],
        dynamic_axes={"input_ids": {0: "batch", 1: "sequence"}, "attention_mask": {0: "batch", 1: "sequence"},
                      "logits": {0: "batch"}},
        opset_version=opset,
        # The TorchScript exporter: its MatMul/Softmax attention pattern is what
        # onnxruntime.transformers.optimizer knows how to fuse.
        dynamo=False,
    )

In [ ]:
%%writefile src/onnxopt/variants.py
"""Turn an fp32 ONNX transformer into the faster variants: graph-optimized and int8-quantized."""

from pathlib import Path


def optimize(fp32: Path, out: Path, num_heads: int, hidden_size: int) -> dict[str, int]:
    """Fuse attention, LayerNorm and GELU into single ONNX Runtime ops; returns the fused op counts.

    opt_level 0 = only the transformer fusions; ONNX Runtime still applies its own CPU
    optimizations when a session loads the file. DistilBERT is BERT-shaped, so model_type="bert".
    """
    from onnxruntime.transformers.optimizer import optimize_model

    model = optimize_model(str(fp32), model_type="bert", num_heads=num_heads, hidden_size=hidden_size, opt_level=0)
    model.save_model_to_file(str(out))
    return {k: v for k, v in model.get_fused_operator_statistics().items() if v}


def quantize_int8(model: Path, out: Path) -> None:
    """Dynamic quantization: int8 weights, activations quantized on the fly (no calibration data)."""
    from onnxruntime.quantization import QuantType, quantize_dynamic
    from onnxruntime.quantization.shape_inference import quant_pre_process

    # Shape inference first, so the quantizer knows every tensor's type around the fused ops.
    prepared = out.with_suffix(".prep.onnx")
    quant_pre_process(str(model), str(prepared), skip_optimization=True)
    try:
        quantize_dynamic(str(prepared), str(out), weight_type=QuantType.QInt8)
    finally:
        prepared.unlink(missing_ok=True)

In [ ]:
%%writefile src/onnxopt/bench.py
"""Timing helpers: run a model over texts in fixed batches and summarise the latencies."""

import time
from collections.abc import Callable

import numpy as np


def run_batches(logits_fn: Callable[[list[str]], np.ndarray], texts: list[str], batch_size: int,
                warmup: int = 3) -> tuple[list[float], np.ndarray]:
    """Time logits_fn on consecutive batches; returns per-batch seconds and all logits in input order."""
    batches = [texts[i:i + batch_size] for i in range(0, len(texts), batch_size)]
    for batch in batches[:warmup]:
        logits_fn(batch)
    seconds, outputs = [], []
    for batch in batches:
        start = time.perf_counter()
        outputs.append(logits_fn(batch))
        seconds.append(time.perf_counter() - start)
    return seconds, np.concatenate(outputs)


def summarize(seconds: list[float], articles: int) -> dict:
    ms = np.array(seconds) * 1000
    return {
        "batches": len(seconds),
        "p50_ms": round(float(np.percentile(ms, 50)), 1),
        "p95_ms": round(float(np.percentile(ms, 95)), 1),
        "mean_ms": round(float(ms.mean()), 1),
        "articles_per_s": round(articles / float(np.sum(seconds)), 1),
    }

In [ ]:
%%writefile scripts/export.py
"""Export the Day 5 DistilBERT to ONNX and check that ONNX Runtime gives the same answers as PyTorch.

    python scripts/export.py

Writes models/model.onnx (+ tokenizer.json, config.json) and results/parity.json.
"""

import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from onnxopt.classifier import OnnxClassifier, encode  # noqa: E402
from onnxopt.data import load_test  # noqa: E402
from onnxopt.export import OPSET, export_onnx  # noqa: E402
from onnxopt.paths import MODELS, ONNX_FILES, RESULTS, SOURCE_MODEL  # noqa: E402

PARITY_ARTICLES = 64


def main() -> None:
    if not SOURCE_MODEL.exists():
        sys.exit(f"error: no model in {SOURCE_MODEL}; set DAY5_MODEL_DIR to the Day 5 models/distilbert folder")
    MODELS.mkdir(exist_ok=True)
    RESULTS.mkdir(exist_ok=True)

    # Eager attention exports as plain MatMul/Softmax ops, which ONNX Runtime's optimizer can fuse later.
    model = AutoModelForSequenceClassification.from_pretrained(SOURCE_MODEL, attn_implementation="eager").eval()
    tokenizer = AutoTokenizer.from_pretrained(SOURCE_MODEL)
    for name in ["tokenizer.json", "config.json"]:
        shutil.copy(SOURCE_MODEL / name, MODELS / name)

    sample = tokenizer(["Ein kurzer Beispielsatz.", "Noch ein etwas längerer Beispielsatz für die Form."],
                       padding=True, return_tensors="pt")
    onnx_path = MODELS / ONNX_FILES["onnx_fp32"]
    start = time.perf_counter()
    export_onnx(model, sample, onnx_path)
    export_seconds = time.perf_counter() - start

    # Parity on real articles, tokenized once by the onnxruntime path (tokenizers only) and fed to both.
    texts, _, _ = load_test()
    texts = texts[:PARITY_ARTICLES]
    onnx = OnnxClassifier(onnx_path)
    hf_ids = tokenizer(texts, truncation=True, max_length=512, padding=True)["input_ids"]
    inputs = encode(onnx.tokenizer, texts)
    same_tokens = hf_ids == inputs["input_ids"].tolist()
    diffs, agree = [], 0
    for i in range(0, len(texts), 8):
        batch = {k: v[i:i + 8] for k, v in inputs.items()}
        with torch.inference_mode():
            ref = model(**{k: torch.from_numpy(v) for k, v in batch.items()}).logits.numpy()
        out = onnx.session.run(["logits"], batch)[0]
        diffs.append(np.abs(ref - out).max())
        agree += int((ref.argmax(-1) == out.argmax(-1)).sum())

    parity = {
        "opset": OPSET,
        "export_seconds": round(export_seconds, 1),
        "articles": len(texts),
        "tokenizers_match_transformers": same_tokens,
        "max_abs_logit_diff": float(max(diffs)),
        "same_label": agree,
        "onnx_mb": round(onnx_path.stat().st_size / 1e6, 1),
        "onnx_files": sorted(p.name for p in MODELS.iterdir()),
    }
    (RESULTS / "parity.json").write_text(json.dumps(parity, indent=2) + "\n")
    print(json.dumps(parity, indent=2))
    if not same_tokens or agree != len(texts) or max(diffs) > 1e-3:
        sys.exit("error: ONNX output differs from PyTorch")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/build_variants.py
"""Build the faster variants from models/model.onnx: graph-optimized and int8-quantized.

    python scripts/export.py            # first
    python scripts/build_variants.py

- onnx_optimized: ONNX Runtime's transformer optimizer fuses attention, LayerNorm and GELU
  into single ops (fewer, bigger kernels).
- onnx_int8: dynamic quantization of the optimized model. Weights are stored as int8; the
  activations are quantized on the fly for each input, so no calibration data is needed.

Writes results/variants.json: file size, fused ops, and how close each variant stays to fp32.
"""

import json
import sys
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from onnxopt.classifier import OnnxClassifier  # noqa: E402
from onnxopt.data import load_test  # noqa: E402
from onnxopt.paths import MODELS, ONNX_FILES, RESULTS  # noqa: E402
from onnxopt.variants import optimize, quantize_int8  # noqa: E402

CHECK_ARTICLES = 64


def main() -> None:
    fp32 = MODELS / ONNX_FILES["onnx_fp32"]
    if not fp32.exists():
        sys.exit("error: run scripts/export.py first")
    optimized = MODELS / ONNX_FILES["onnx_optimized"]
    int8 = MODELS / ONNX_FILES["onnx_int8"]

    config = json.loads((MODELS / "config.json").read_text())
    fused = optimize(fp32, optimized, num_heads=config["n_heads"], hidden_size=config["dim"])
    quantize_int8(optimized, int8)

    texts, _, _ = load_test()
    texts = texts[:CHECK_ARTICLES]
    ref = OnnxClassifier(fp32).logits(texts)
    report = {"check_articles": len(texts), "fused_operators": fused, "variants": {}}
    for name, file in ONNX_FILES.items():
        logits = ref if name == "onnx_fp32" else OnnxClassifier(MODELS / file).logits(texts)
        report["variants"][name] = {
            "file": file,
            "mb": round((MODELS / file).stat().st_size / 1e6, 1),
            "max_abs_logit_diff_vs_fp32": float(np.abs(logits - ref).max()),
            "same_label_as_fp32": int((logits.argmax(-1) == ref.argmax(-1)).sum()),
        }
    (RESULTS / "variants.json").write_text(json.dumps(report, indent=2) + "\n")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile scripts/benchmark.py
"""Benchmark every variant: accuracy on all 1,028 test articles, then latency and throughput.

    python scripts/benchmark.py                   # all phases (about 45 min on an M2 Air)
    python scripts/benchmark.py --phase timing    # re-time only; keeps saved accuracy runs
    python scripts/benchmark.py --phase summary   # rebuild benchmark.json from the saved runs
    python scripts/benchmark.py --out results/colab_x86 --threads 2 --timing-articles 128 \
        --rounds 2 --cooldown 0 --accuracy-variants onnx_int8   # the Colab x86 run

Accuracy: each variant classifies all 1,028 test articles at batch sizes 1, 8 and 32 (padding
can change int8 results, because dynamic quantization scales each batch as a whole).

Timing: 3 rounds on the first 256 test articles. The variant order rotates every round and
there is a cool-down between variants, because a fanless laptop slows down under sustained
load and would otherwise penalise whichever variant runs last. Every variant gets the same
thread budget, the same tokenizer (timed as part of each call) and a short warm-up.

Each run happens in its own process, so peak memory is measured per variant.
Writes results/benchmark/*.json (raw runs) and results/benchmark.json (summary).
"""

import argparse
import json
import platform
import resource
import subprocess
import sys
import time
from pathlib import Path

import numpy as np

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))
from onnxopt.bench import run_batches, summarize  # noqa: E402
from onnxopt.classifier import encode, load_tokenizer  # noqa: E402
from onnxopt.data import load_test  # noqa: E402
from onnxopt.paths import MODELS, ONNX_FILES, RESULTS, SOURCE_MODEL, VARIANTS  # noqa: E402

BATCH_SIZES = [1, 8, 32]
THREADS = 4  # default: the M2's 4 performance cores (set with --threads)


def load_variant(name: str):
    """Returns (logits_fn, load_seconds, size_mb)."""
    start = time.perf_counter()
    if name == "pytorch":
        import torch
        from transformers import AutoModelForSequenceClassification

        torch.set_num_threads(THREADS)
        model = AutoModelForSequenceClassification.from_pretrained(SOURCE_MODEL).eval()
        tokenizer = load_tokenizer(MODELS)

        def logits_fn(texts):
            with torch.inference_mode():
                inputs = {k: torch.from_numpy(v) for k, v in encode(tokenizer, texts).items()}
                return model(**inputs).logits.numpy()

        size = (SOURCE_MODEL / "model.safetensors").stat().st_size
    else:
        from onnxopt.classifier import OnnxClassifier

        clf = OnnxClassifier(MODELS / ONNX_FILES[name], threads=THREADS)
        logits_fn = clf.logits
        size = (MODELS / ONNX_FILES[name]).stat().st_size
    return logits_fn, time.perf_counter() - start, round(size / 1e6, 1)


def peak_rss_mb() -> float:
    rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    return round(rss / 1e6 if sys.platform == "darwin" else rss * 1024 / 1e6, 1)  # bytes on macOS, KiB on Linux


def accuracy_run(name: str) -> dict:
    from sklearn.metrics import accuracy_score, f1_score

    texts, labels, _ = load_test()
    logits_fn, load_seconds, size_mb = load_variant(name)
    out = {"variant": name, "size_mb": size_mb, "load_seconds": round(load_seconds, 2), "batch_sizes": {}}
    for batch_size in BATCH_SIZES:
        _, logits = run_batches(logits_fn, texts, batch_size, warmup=0)
        pred = logits.argmax(-1)
        out["batch_sizes"][str(batch_size)] = {
            "accuracy": round(accuracy_score(labels, pred), 4),
            "macro_f1": round(f1_score(labels, pred, average="macro"), 4),
            "predictions": pred.tolist(),
        }
    out["peak_rss_mb"] = peak_rss_mb()
    return out


def timing_run(name: str, articles: int) -> dict:
    texts = load_test()[0][:articles]
    logits_fn, _, _ = load_variant(name)
    seconds = {str(b): run_batches(logits_fn, texts, b)[0] for b in BATCH_SIZES}
    return {"variant": name, "articles": len(texts), "seconds": seconds}


def machine() -> dict:
    import onnxruntime
    import torch

    cpu = platform.processor()
    if sys.platform == "darwin":
        cpu = subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"], capture_output=True, text=True).stdout.strip()
    elif Path("/proc/cpuinfo").exists():
        names = [line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines()
                 if line.startswith("model name")]
        cpu = f"{names[0]} ({len(names)} vCPUs)" if names else cpu
    return {"cpu": cpu, "os": platform.platform(), "threads": THREADS, "python": platform.python_version(),
            "onnxruntime": onnxruntime.__version__, "torch": torch.__version__}


def summarize_all(folder: Path, args) -> dict:
    variants = {}
    reference = None
    for name in VARIANTS:
        acc_file = folder / f"accuracy_{name}.json"
        acc = json.loads(acc_file.read_text()) if acc_file.exists() else None
        if name == "pytorch" and acc:
            reference = np.array(acc["batch_sizes"]["1"]["predictions"])
        rounds = [json.loads(p.read_text()) for p in sorted(folder.glob(f"timing_{name}_r*.json"))]
        size = SOURCE_MODEL / "model.safetensors" if name == "pytorch" else MODELS / ONNX_FILES[name]
        out = {"size_mb": round(size.stat().st_size / 1e6, 1), "rounds": len(rounds)}
        if acc:
            out.update(load_seconds=acc["load_seconds"], peak_rss_mb=acc["peak_rss_mb"])
        out["batch_sizes"] = {}
        for batch in map(str, BATCH_SIZES):
            pooled = [s for r in rounds for s in r["seconds"][batch]]
            stats = summarize(pooled, sum(r["articles"] for r in rounds))
            stats["articles_per_s_by_round"] = [round(r["articles"] / sum(r["seconds"][batch]), 1) for r in rounds]
            if acc:
                pred = np.array(acc["batch_sizes"][batch]["predictions"])
                stats.update(accuracy=acc["batch_sizes"][batch]["accuracy"], macro_f1=acc["batch_sizes"][batch]["macro_f1"])
                if reference is not None:
                    stats["same_label_as_pytorch"] = int((pred == reference).sum())
            out["batch_sizes"][batch] = stats
        variants[name] = out
    return {"machine": machine(), "timing_articles": args.timing_articles, "timing_rounds": args.rounds,
            "cooldown_seconds": args.cooldown, "variants": variants}


def main() -> None:
    global THREADS  # load_variant reads it; set from --threads
    parser = argparse.ArgumentParser()
    parser.add_argument("--phase", choices=["all", "accuracy", "timing", "summary"], default="all")
    parser.add_argument("--variant", choices=VARIANTS, help="internal: run one variant in this process")
    parser.add_argument("--round", type=int, default=1, help="internal: timing round number")
    parser.add_argument("--out", type=Path, default=RESULTS, help="results folder")
    parser.add_argument("--threads", type=int, default=THREADS)
    parser.add_argument("--timing-articles", type=int, default=256)
    parser.add_argument("--rounds", type=int, default=3)
    parser.add_argument("--cooldown", type=float, default=20, help="seconds of rest between runs")
    parser.add_argument("--accuracy-variants", nargs="+", choices=VARIANTS, default=VARIANTS)
    args = parser.parse_args()
    THREADS = args.threads
    folder = args.out / "benchmark"
    folder.mkdir(parents=True, exist_ok=True)
    shared = ["--out", str(args.out), "--threads", str(args.threads), "--timing-articles", str(args.timing_articles)]

    def run_child(*child_args: str) -> None:
        subprocess.run([sys.executable, __file__, *shared, *child_args], check=True)
        time.sleep(args.cooldown)

    if args.variant:
        if args.phase == "accuracy":
            (folder / f"accuracy_{args.variant}.json").write_text(json.dumps(accuracy_run(args.variant)) + "\n")
        else:
            result = timing_run(args.variant, args.timing_articles)
            (folder / f"timing_{args.variant}_r{args.round}.json").write_text(json.dumps(result) + "\n")
            b1 = result["articles"] / sum(result["seconds"]["1"])
            print(f"round {args.round}  {args.variant:15s} batch 1: {b1:5.1f} articles/s", flush=True)
        return

    if args.phase in ("all", "accuracy"):
        for name in args.accuracy_variants:
            run_child("--phase", "accuracy", "--variant", name)
    if args.phase in ("all", "timing"):
        for old in folder.glob("timing_*.json"):
            old.unlink()
        for r in range(args.rounds):
            order = VARIANTS[r:] + VARIANTS[:r]  # rotate, so no variant always runs last (hottest)
            for name in order:
                run_child("--phase", "timing", "--variant", name, "--round", str(r + 1))
    (args.out / "benchmark.json").write_text(json.dumps(summarize_all(folder, args), indent=2) + "\n")
    print(f"wrote {args.out / 'benchmark.json'}")


if __name__ == "__main__":
    main()

Upload the Day 5 model (about 250 MB):

In [ ]:
from google.colab import files
files.upload()
!unzip -qo day5_distilbert.zip -d day5 && find day5 -name config.json
assert _exit_code == 0, "this step failed: see the error above"

In [ ]:
import glob, os
os.environ["DAY5_MODEL_DIR"] = os.path.dirname(glob.glob("day5/**/model.safetensors", recursive=True)[0])
!python scripts/export.py
assert _exit_code == 0, "this step failed: see the error above"
!python scripts/build_variants.py
assert _exit_code == 0, "this step failed: see the error above"

Benchmark: 2 threads (Colab's 2 vCPUs), 128 timing articles x 2 rounds, int8 accuracy on all 1,028 articles.

In [ ]:
!python scripts/benchmark.py --out results/colab_x86 --threads 2 --timing-articles 128 --rounds 2 --cooldown 0 --accuracy-variants onnx_int8
assert _exit_code == 0, "this step failed: see the error above"

In [ ]:
!cp results/parity.json results/variants.json results/colab_x86/
!zip -qr onnx_x86_results.zip results/colab_x86
files.download("onnx_x86_results.zip")